# Lab 16 · Integrated pgvector RAG application
## Goal
Operate the stack as one application: ingest an update, query it through FastAPI, verify tenant isolation, collect an evaluation report, and diagnose a controlled failure. Prerequisites: Labs 12–15. Plan two 90-minute sessions or a capstone assignment; full deployment and evaluation are not compressed into one short tutorial.

pgvector remains the required backend. The earlier optional Milvus/Pinecone portability proposal is retained as an extension at the end; the core practical work goes deeper into the selected database and FastAPI stack.

## Setup
Complete the track README preflight first: install this track's pinned environment and editable package; start PostgreSQL/pgvector with Docker Compose; download the two pinned model snapshots; seed the invented course corpus. Select that Python environment as the kernel. These notebooks need the **whole `labs/rag` folder**, not a standalone upload. They use a real database and local neural models. Missing services or models are errors, not silently replaced by mocks.

Run worked cells first, then implement TODOs, set `RUN_EXERCISES=True`, restart, and run all. Instructor copies enable those checks. `RUN_GENERATION` stays false unless you deliberately enable the separately documented Gemini request. Offline here means no inference API calls after package/model downloads; PostgreSQL is still a real local service. All data are invented and English-language. Never use this teaching database or its public example keys for real student records.

In [1]:
import json
from pathlib import Path
from importlib.metadata import version
import numpy as np
from raglab.config import ROOT, DIMENSION
from raglab import db, models
from raglab.ingestion import corpus, chunk_documents, make_pipeline
RUN_EXERCISES = True
RUN_GENERATION = False
print({p:version(p) for p in ("llama-index-core","pgvector","psycopg","fastapi","sentence-transformers")})
with db.connect() as conn:
    print(conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone())

{'llama-index-core': '0.14.25', 'pgvector': '0.5.0', 'psycopg': '3.3.6', 'fastapi': '0.142.4', 'sentence-transformers': '6.1.0'}
{'extversion': '0.8.1'}


## Steps
### 1. Verify a database update through the API
A database write is not enough: verify the service returns the current version's evidence and that old chunks were removed. Use a dedicated exercise tenant so the evaluation corpus is unchanged. Each change is a transaction; embedding computation happens before the transaction begins.

In [2]:
from raglab.service import create_app
from fastapi.testclient import TestClient
record={"tenant":"exercise16","doc_id":"policy","version":1,"title":"Demonstration policy","body":"The capstone camera loan lasts three days.","source":"course://exercise16/policy"}
def store(record):
    chunks=chunk_documents([record])
    db.replace_document(record,chunks,models.embed([c["text"] for c in chunks]))
try:
    store(record)
    app=create_app({"capstone-key":"exercise16"})
    with TestClient(app) as client:
        payload={"query":"camera loan","mode":"lexical","k":5}
        first=client.post("/search",headers={"X-API-Key":"capstone-key"},json=payload).json()
        store(dict(record,version=2,body="The capstone camera loan lasts six days."))
        second=client.post("/search",headers={"X-API-Key":"capstone-key"},json=payload).json()
        assert any("six days" in h["text"] for h in second["hits"])
        assert not any("three days" in h["text"] for h in second["hits"])
        assert {h["chunk_id"] for h in first["hits"]}.isdisjoint(h["chunk_id"] for h in second["hits"])
        print({"before":first["hits"],"after":second["hits"]})
finally:
    db.delete_document("exercise16","policy")

/private/tmp/course-rag/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
/private/tmp/course-rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11376.26it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6497.90it/s]

{'before': [{'chunk_id': '7a9be5540adae23b37c929d25b82ceda0a248641931a68f9ef6df125f8c26c3a', 'doc_id': 'policy', 'text': 'The capstone camera loan lasts three days.'}], 'after': [{'chunk_id': 'e25cf9fc065c038836376369851f93afc2a0c91690fde04ed133bd5787ed0047', 'doc_id': 'policy', 'text': 'The capstone camera loan lasts six days.'}]}


### 2. TODO: implement a deployment smoke test (30 points)
Implement `smoke(client)` against the reference FastAPI app. Verify health, authenticated alpha retrieval, beta retrieval of its marker, rejection of a missing key, and alpha isolation. Return a dictionary of named boolean checks only after asserting all are true. Use test-client responses now; repeat the same checks against an `httpx.Client` connected to the real server before submission.

In [3]:
def smoke(client):
    body={"query":"BETA-ONLY-MARKER camera","k":20}
    a=client.post("/search",headers={"X-API-Key":"student-alpha"},json=body)
    b=client.post("/search",headers={"X-API-Key":"student-beta"},json=body)
    checks={"health":client.get("/health").status_code==200,
        "alpha_search":a.status_code==200,
        "beta_search":b.status_code==200 and any("BETA-ONLY-MARKER" in h["text"] for h in b.json()["hits"]),
        "isolation":a.status_code==200 and all("BETA-ONLY-MARKER" not in h["text"] for h in a.json()["hits"]),
        "auth":client.post("/search",json=body).status_code==401}
    assert all(checks.values()),checks
    return checks

In [4]:
if RUN_EXERCISES:
    with TestClient(create_app({"student-alpha":"alpha","student-beta":"beta"})) as client:
        print(smoke(client))

{'health': True, 'alpha_search': True, 'beta_search': True, 'isolation': True, 'auth': True}


### 3. TODO: summarize evaluation without hiding missing labels (30 points)
Implement `summarize(rows)` for records from `raglab.evaluate.evaluate`. Group by method, report the number of query runs, number with non-null relevance metrics, mean Recall@5 over labeled queries, and median observed milliseconds over all runs. Empty input returns `{}`. Do not turn null recall into zero or include it in the relevance denominator. Reject negative/nonfinite time or recall outside [0,1].

In [5]:
def summarize(rows):
    groups={}
    for row in rows:
        t=row["milliseconds"]; r=row["recall"]
        if not np.isfinite(t) or t<0 or (r is not None and (not np.isfinite(r) or not 0<=r<=1)):
            raise ValueError("invalid measurement")
        groups.setdefault(row["mode"],[]).append(row)
    out={}
    for mode,group in groups.items():
        values=[r["recall"] for r in group if r["recall"] is not None]
        out[mode]={"queries":len(group),"labeled":len(values),"mean_recall":float(np.mean(values)) if values else None,
                   "median_ms":float(np.median([r["milliseconds"] for r in group]))}
    return out

In [6]:
if RUN_EXERCISES:
    assert summarize([])=={}
    example=[{"mode":"dense","milliseconds":2,"recall":1.0},{"mode":"dense","milliseconds":4,"recall":None}]
    assert summarize(example)["dense"]=={"queries":2,"labeled":1,"mean_recall":1.0,"median_ms":3.0}
    for bad in (-1,float("nan"),float("inf")):
        try: summarize([{"mode":"x","milliseconds":bad,"recall":None}])
        except ValueError: pass
        else: raise AssertionError("invalid timing accepted")

### 4. Optional Gemini generation from service evidence
This completes a retrieval→generation path when you have a Gemini account. Retrieval practice and grading do not require a paid API. Set `GEMINI_API_KEY` and `GEMINI_MODEL` outside the notebook, then deliberately enable `RUN_GENERATION`. The helper performs one model request with a timeout and no automatic retry beyond that attempt. Citation membership is checked; entailment and answer completeness still require separate assessment.

In [7]:
if RUN_GENERATION:
    from raglab.generation import grounded_answer
    with TestClient(create_app({"student-alpha":"alpha"})) as client:
        query="How long can I borrow a camera?"
        response=client.post("/search",headers={"X-API-Key":"student-alpha"},json={"query":query,"k":5,"rerank":True})
        response.raise_for_status()
        answer=grounded_answer(query,response.json()["hits"])
        print(answer.model_dump())
else:
    print("Gemini generation not executed; no generated-answer quality claim.")

Gemini generation not executed; no generated-answer quality claim.


### 5. Package and operate the stack
From `labs/rag`, after preparing model snapshots and seeding PostgreSQL:

```bash
docker compose -f compose.yaml -f compose.api.yaml up --build -d --wait
curl -sS http://127.0.0.1:58000/health
```

The API image mounts the already-downloaded model snapshots read-only. Record the built image ID, model revisions, package lock, and database version. Run the smoke test against `httpx.Client(base_url="http://127.0.0.1:58000")`. Stop only this stack with `docker compose -f compose.yaml -f compose.api.yaml down`; retain the database volume unless you intentionally want to erase the teaching data.

Freeze settings before running `python -m raglab.evaluate --split test --output student-results/final.json`. The test questions are public; use them for disciplined practice, not as a claim of blind evaluation. Latency in the evaluation report includes cold-start effects and is not a concurrent-load benchmark.

## Checks and submission
Submit app code, ten or more meaningful tests, one query-level evaluation JSON, a short operating guide, and a failure report. Demonstrate stale-update rejection, unknown tenant, unauthorized request, empty lexical results, and a simulated database timeout. Explain which failures return 503 and which should fail startup. Include a trace from source document to retrieved chunk to cited evidence.

Rubric: smoke checks 30, evaluation summary 30, deployment and failure tests 25, reproducibility and interpretation 15. For generated answers, assess evidence support and abstention separately from retrieval relevance. Raw neural scores are not universal abstention thresholds.

## Next Steps · Optional backend portability
Keep the same query corpus, embeddings, tenant policy, and evaluation contract. Implement one alternative adapter using Milvus or Pinecone and compare filtered retrieval, ingestion/update semantics, returned IDs, setup requirements, and measured cost/latency under declared conditions. Do not replace the required pgvector implementation. A managed API and a local container are not directly comparable performance environments.

References checked 2026-10-07: [Docker Compose startup order](https://docs.docker.com/compose/how-tos/startup-order/), [FastAPI testing](https://fastapi.tiangolo.com/tutorial/testing/), [Gemini structured output](https://ai.google.dev/gemini-api/docs/structured-output), [Milvus hybrid search](https://milvus.io/docs/multi-vector-search.md), [Pinecone hybrid search](https://docs.pinecone.io/guides/search/hybrid-search).